# 📚 LangChain ile Retrieval Augmented Generation'a Giriş 🦜🔗

Bu not defterinde LangChain kullanarak Retrieval Augmented Generation'ı nasıl kullanacağınızı öğreneceksiniz.

Kendi belgelerimiz hakkında sorular sormak için bir LLM kullanacağız!

## ⚙️ Kurulum

👉 Temel kütüphaneleri içe aktarmak için aşağıdaki hücreyi çalıştırın.

In [3]:
%load_ext autoreload
%autoreload 2
import os
from pprint import pprint
from IPython.display import Markdown

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


👉 API anahtarımızı tekrar yüklemek için aşağıdaki hücreyi çalıştırın:

In [4]:
from dotenv import load_dotenv

load_dotenv()  # Load environment variables from .env file

True

## 📚 Neden RAG?

Bir LLM kendi başına, öğrendiği her şey hakkında sorulara yanıt verebilir.

Bunun birkaç dezavantajı vardır:
- Eğitim verileri geçmişten gelir ve en son verilerle güncellenmez.
- Sadece eğitim aldığı verileri bilir.

Bir LLM'yi kendi verilerimizle çalışması için kullanmak istiyoruz. İşte bu noktada RAG (Retrieval-Augmented Generation) devreye girer.

1. **Retrieval-Augmented Generation (RAG)**, gerçek doğruluğu artırmak için bir dil modelini belge alıcı ile birleştirir.
2. **İlgili dış belgeleri alır** (örneğin, bilgi tabanından) yanıtlar üretmeden önce.
3. **Dil modeli hem istemi hem de alınan bağlamı kullanarak** daha bilgili ve temelli çıktılar üretir.

## 🇪🇺 Bağlam

Bu meydan okumada, Avrupa Parlamentosu'ndan belgelerle çalışacağız.

Bir gazeteci olduğunuzu ve Avrupa Parlamentosu'nun genel kurul oturumları sırasında belirli bir konu hakkında neler söylendiğini öğrenmek istediğinizi düşünün. Bu oturumlar yılda 12 kez Strasbourg'da gerçekleşir ve 4 gün sürer. Oturumların transkriptleri EP'nin web sitesinde mevcuttur.

Kesinlikle tüm bu transkriptleri karıştırmak istemezsiniz. O halde, hayatımızı kolaylaştırmak için RAG'ı kullanalım!

Bu, her zaman test etmek için yepyeni veriler alabileceğimiz için çalışmak üzere iyi verilerdir.

## 📘 Verileri alalım

1. [EP'nin web sitesine](https://www.europarl.europa.eu/plenary/en/debates-video.html) gidin. 
1. Bu sizi en son genel kurul oturumuna yönlendirecektir.
1. İlk tarihin altında, "▶️ Verbatim reports HTML" bölümünde `HTML`'e tıklayın.
1. Sayfanın sonuna kaydırın ve alttaki PDF dosyasını indirin.
1. Dosyayı `data` klasörüne kaydedin.

Bir belgeyle başlayacağız, ancak daha sonrası için diğer birkaç günün aynısını şimdiden indirebilirsiniz.

Belgeye bir göz atın. Kaç sayfası var? Belge hakkında bir fikir edinmek için hızlıca belgede gezinin.

## 🔢 Belgeleri gömme

Belgeleri gömmek, tüm belgeleri veya belge parçalarını vektörlere çevirmek anlamına gelir.

LangChain🦜🔗 yine çok yardımcı olacak.

Bir gömme aracı (embedder) başlatalım ve deneyelim. LLM olarak Gemini kullandığımız için, Google'ın metin gömme araçlarında kalalım.

In [5]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

👉 Basit bir metin parçasını gömmek için gömme aracının `.embed_query()` metodunu deneyin.

In [6]:
sample_embedding = embeddings.embed_query("What is the capital of France?")

👉 Bu `sample_embedding`'i keşfetmek için zaman ayırın. Nasıl görünüyor? Tipi nedir? Gömme boyutu nedir?

In [7]:
print(type(sample_embedding))
print(len(sample_embedding))
print(sample_embedding[:5])

<class 'list'>
3072
[-0.032554302, 0.01305496, 0.015067407, -0.07128718, -0.031062366]


## 💾 PDF'den gerçek verilerimizi yükle

Artık bir gömmenin nasıl göründüğünü biliyoruz, gerçek verilerimizle çalışmanın zamanı geldi.

👉 [LangChain belgelerine](https://docs.langchain.com/oss/python/integrations/document_loaders/index#pdfs) gidin ve PyPDF kullanarak bir PDF'yi nasıl yükleyebileceğinizi öğrenin.

👉 Sonra devam edin ve daha önce indirdiğiniz PDF'lerden birini yükleyin.

In [8]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("data/CRE-10-2026-05-18_EN.pdf")
pages = loader.load()

👉 `pages`'i keşfedin:
- Veri tipi nedir?
- Kaç sayfanız var?
- Bir sayfanın tipi nedir?
- Bir sayfanın içeriğine nasıl erişebilirsiniz?
- Tam belgenin kaç karakteri var?
- Bir sayfanın `metadata`'sında neler var?

In [9]:
print(type(pages))
print(len(pages))
print(type(pages[0]))
print(pages[0].page_content[:300])
print(len(''.join([p.page_content for p in pages])))
print(pages[0].metadata)

<class 'list'>
123
<class 'langchain_core.documents.base.Document'>
2024-2029 
 
 
ПЪЛЕН ПРОТОКОЛ НА РАЗИСКВАНИЯТА  DEBAŠU STENOGRAMMA 
ACTA LITERAL DE LOS DEBATES  POSĖDŽIO STENOGRAMA 
DOSLOVNÝ ZÁZNAM ZE ZASEDÁNÍ  AZ ÜLÉSEK SZÓ SZERINTI JEGYZŐKÖNYVE 
FULDSTÆNDIGT FORHANDLINGSREFERAT  RAPPORTI VERBATIM TAD-DIBATTITI 
AUSFÜHRLICHE SITZUNGSBERICHTE  VOLLEDIG VERSLAG V
374979
{'producer': 'Aspose.Words for Java 24.2.0', 'creator': 'Aspose.Words', 'creationdate': '', 'author': 'e-Parliament@europarl.europa.eu', 'dmxml.render.id': '511458', 'dmxml.render.traceid': '6a198204812b310a18e00bd9e931b1b9', 'uid': 'eu.europa.europarl-DIN1-2026-0000154886_01.00-xm-01.00_text-xml', 'source': 'data/CRE-10-2026-05-18_EN.pdf', 'total_pages': 123, 'page': 0, 'page_label': '1'}


## ✂️ Verilerimizi böl

Tam belgemiz gömülmek için çok uzun. Metin gömme aracımız 2.048 tokena kadar giriş alabilir. Gemini modelleri için bu yaklaşık 8.196 karakterdir (token başına 4 karakter).

Bu yüzden belgemizi daha küçük parçalara bölmek istiyoruz.

Zaten çalışabileceğimiz bir dizi sayfamız var. Ama sayfa sonları biraz keyfi: genellikle cümlenin ortasında görünürler.

Ayrıca, sayfalar arasında örtüşme yoktur. Bu yüzden bir sayfanın ilk satırı önceki tüm bağlamı kaçırır. Tam metni biraz örtüşmeyle bölmek daha iyidir.

İlk olarak, PDF'yi tekrar yükleyeceğiz, bu sefer bölmeden.

In [10]:
file_path = "data/CRE-10-2026-05-18_EN.pdf"
loader = PyPDFLoader(file_path, mode='single')
pdf = loader.load()
pdf_text = pdf[0].page_content
len(pdf_text)

375223

Artık tüm PDF'imizi tek bir belge olarak aldığımıza göre, onu daha akıllı bir şekilde parçalara bölebiliriz.

👉 Yine, ["Özyinelemeli olarak bölme" konusundaki LangChain belgelerine](https://docs.langchain.com/oss/python/integrations/splitters/recursive_text_splitter) gidin ve `pdf` _belgelerimizi_ parçalara (LangChain'de `documents` olarak adlandırılır) nasıl böleceğinizi öğrenin.

2_000 karakter (bizim durumumuzda yaklaşık yarım sayfa) parçalara 400 örtüşmeyle bölün. İsterseniz diğer değerlerle deneyebilirsiniz.

`RecursiveCharacterTextSplitter`'ın `.split_documents()` metodunu kullanın: bu metod giriş olarak bir belge alır ve bölünmüş belgeler çıktısı verir.

In [11]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=400)
all_splits = splitter.split_documents(pdf)

👉 `all_splits`'i inceleyin:
- Veri tipi nedir?
- Kaç bölümünüz var?
- Bir bölümün tipi nedir?
- Bir bölümün içeriğine nasıl erişebilirsiniz?
- Şimdi toplamda kaç karakterimiz var?
- Bir bölümün `metadata`'sında neler var?

In [12]:
print(type(all_splits))
print(len(all_splits))
print(type(all_splits[0]))
print(all_splits[0].page_content[:300])
print(sum([len(s.page_content) for s in all_splits]))
print(all_splits[0].metadata)

<class 'list'>
235
<class 'langchain_core.documents.base.Document'>
2024-2029 
 
 
ПЪЛЕН ПРОТОКОЛ НА РАЗИСКВАНИЯТА  DEBAŠU STENOGRAMMA 
ACTA LITERAL DE LOS DEBATES  POSĖDŽIO STENOGRAMA 
DOSLOVNÝ ZÁZNAM ZE ZASEDÁNÍ  AZ ÜLÉSEK SZÓ SZERINTI JEGYZŐKÖNYVE 
FULDSTÆNDIGT FORHANDLINGSREFERAT  RAPPORTI VERBATIM TAD-DIBATTITI 
AUSFÜHRLICHE SITZUNGSBERICHTE  VOLLEDIG VERSLAG V
457861
{'producer': 'Aspose.Words for Java 24.2.0', 'creator': 'Aspose.Words', 'creationdate': '', 'author': 'e-Parliament@europarl.europa.eu', 'dmxml.render.id': '511458', 'dmxml.render.traceid': '6a198204812b310a18e00bd9e931b1b9', 'uid': 'eu.europa.europarl-DIN1-2026-0000154886_01.00-xm-01.00_text-xml', 'source': 'data/CRE-10-2026-05-18_EN.pdf', 'total_pages': 123}


## 🗄️ Her şeyi bir araya getir: belgelerimizi gömme ve vektör deposunda sakla

Elimizde şunlar var:
- Bir gömme aracı
- Veriyi yüklemek için bir yükleyici
- Belgemizi belgelere bölmek için bir metin bölücü

Neyi kaçırıyoruz?

Belgelerimizi gömebiliriz, ama onları bir yerde saklamak istiyoruz. İşte burada vektör deposu devreye girer: şunları saklamamıza olanak sağlar:
- belgeyi (parçayı),
- onun gömmesini,
- meta verilerini.

Sonraki adımda belgeleri verimli bir şekilde alabilecek olacağız.

👉 Bir `InMemoryVectorStore` nasıl oluşturabileceğinizi görmek için ["Vektör depoları" üzerine LangChain belgelerini](https://docs.langchain.com/oss/python/langchain/knowledge-base#3-vector-stores) kontrol edin.

In [20]:
embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-2",
    google_api_key=os.getenv("GOOGLE_API_KEY")
)

In [22]:
import time
from langchain_core.vectorstores import InMemoryVectorStore

vector_store = InMemoryVectorStore(embedding=embeddings)

batch_size = 5
document_ids = []

for i in range(0, len(all_splits), batch_size):
    batch = all_splits[i:i+batch_size]
    ids = vector_store.add_documents(documents=batch)
    document_ids.extend(ids)
    print(f"{i+batch_size}/{len(all_splits)} tamamlandı")
    time.sleep(10)

5/235 tamamlandı
10/235 tamamlandı
15/235 tamamlandı
20/235 tamamlandı
25/235 tamamlandı
30/235 tamamlandı
35/235 tamamlandı
40/235 tamamlandı
45/235 tamamlandı
50/235 tamamlandı
55/235 tamamlandı
60/235 tamamlandı
65/235 tamamlandı
70/235 tamamlandı
75/235 tamamlandı
80/235 tamamlandı
85/235 tamamlandı
90/235 tamamlandı
95/235 tamamlandı
100/235 tamamlandı
105/235 tamamlandı
110/235 tamamlandı
115/235 tamamlandı
120/235 tamamlandı
125/235 tamamlandı
130/235 tamamlandı
135/235 tamamlandı
140/235 tamamlandı
145/235 tamamlandı
150/235 tamamlandı
155/235 tamamlandı
160/235 tamamlandı
165/235 tamamlandı
170/235 tamamlandı
175/235 tamamlandı
180/235 tamamlandı
185/235 tamamlandı
190/235 tamamlandı
195/235 tamamlandı
200/235 tamamlandı
205/235 tamamlandı
210/235 tamamlandı
215/235 tamamlandı
220/235 tamamlandı
225/235 tamamlandı
230/235 tamamlandı
235/235 tamamlandı


In [23]:
print(document_ids[:3])

['4de85666-f66e-49e9-a9f1-242393990ff9', 'b39d3f12-26e0-41d8-a0b9-55e0ac5b2df7', '17d928e5-4009-4b89-95c9-d5bd08c158ee']


In [24]:
vector_store.get_by_ids(document_ids[:3])

[Document(id='4de85666-f66e-49e9-a9f1-242393990ff9', metadata={'producer': 'Aspose.Words for Java 24.2.0', 'creator': 'Aspose.Words', 'creationdate': '', 'author': 'e-Parliament@europarl.europa.eu', 'dmxml.render.id': '511458', 'dmxml.render.traceid': '6a198204812b310a18e00bd9e931b1b9', 'uid': 'eu.europa.europarl-DIN1-2026-0000154886_01.00-xm-01.00_text-xml', 'source': 'data/CRE-10-2026-05-18_EN.pdf', 'total_pages': 123}, page_content='2024-2029 \n \n \nПЪЛЕН ПРОТОКОЛ НА РАЗИСКВАНИЯТА  DEBAŠU STENOGRAMMA \nACTA LITERAL DE LOS DEBATES  POSĖDŽIO STENOGRAMA \nDOSLOVNÝ ZÁZNAM ZE ZASEDÁNÍ  AZ ÜLÉSEK SZÓ SZERINTI JEGYZŐKÖNYVE \nFULDSTÆNDIGT FORHANDLINGSREFERAT  RAPPORTI VERBATIM TAD-DIBATTITI \nAUSFÜHRLICHE SITZUNGSBERICHTE  VOLLEDIG VERSLAG VAN DE VERGADERINGEN \nISTUNGI STENOGRAMM  PEŁNE SPRAWOZDANIE Z OBRAD \nΠΛΗΡΗ ΠΡΑΚΤΙΚΑ ΤΩΝ ΣΥΖΗΤΗΣΕΩΝ  RELATO INTEGRAL DOS DEBATES \nVERBATIM REPORT OF PROCEEDINGS STENOGRAMA DEZBATERILOR \nCOMPTE RENDU IN EXTENSO DES DÉBATS  DOSLOVNÝ ZÁPIS Z ROZPRÁV \nT

👉 Bir vektör deposundaki belgenin içeriğine ve meta verilerine nasıl erişebilirsiniz?

In [25]:
docs = vector_store.get_by_ids(document_ids[:3])
for doc in docs:
    print(doc.page_content[:100])
    print(doc.metadata)
    print("---")

2024-2029 
 
 
ПЪЛЕН ПРОТОКОЛ НА РАЗИСКВАНИЯТА  DEBAŠU STENOGRAMMA 
ACTA LITERAL DE LOS DEBATES  POS
{'producer': 'Aspose.Words for Java 24.2.0', 'creator': 'Aspose.Words', 'creationdate': '', 'author': 'e-Parliament@europarl.europa.eu', 'dmxml.render.id': '511458', 'dmxml.render.traceid': '6a198204812b310a18e00bd9e931b1b9', 'uid': 'eu.europa.europarl-DIN1-2026-0000154886_01.00-xm-01.00_text-xml', 'source': 'data/CRE-10-2026-05-18_EN.pdf', 'total_pages': 123}
---
Unida na diversidade - Unită în diversitate - Zjednotení v rozmanitosti - Združena v raznolikosti - 
{'producer': 'Aspose.Words for Java 24.2.0', 'creator': 'Aspose.Words', 'creationdate': '', 'author': 'e-Parliament@europarl.europa.eu', 'dmxml.render.id': '511458', 'dmxml.render.traceid': '6a198204812b310a18e00bd9e931b1b9', 'uid': 'eu.europa.europarl-DIN1-2026-0000154886_01.00-xm-01.00_text-xml', 'source': 'data/CRE-10-2026-05-18_EN.pdf', 'total_pages': 123}
---
8 - Състав на комисиите и делегациите ..........................

## 🔎 Benzer belgeleri almak için vektör deposunu kullan

Artık belgeleri gömleğe çevirdiğimize göre, benzer belgeleri almak için vektör deposunu kullanabiliriz.

👉 Bunun nasıl çalıştığını görmek için ["Vektör depoları" üzerine LangChain belgelerini](https://docs.langchain.com/oss/python/langchain/knowledge-base#3-vector-stores) kontrol edin.

Bir sorgu kullanın, örneğin "Tarım politikası üzerine tartışmayı özetle.", ve en benzer belgeleri bulun. Ayrıca alınacak belge sayısını da belirtebilirsiniz.

In [26]:
query = "Discuss the agricultural policy"
retrieved_docs = vector_store.similarity_search(query, k=4)


Bu, RAG'ın sözde "Alma" (Retrieval) kısmını tamamlar: artık sorgumuza en benzer belgeleri bulabiliriz.

Çalışmanın çoğu artık tamamlandı!

## 💬 Sorumuza bir cevap üret

Şimdiye kadar benzer belgeleri almamızı sağlamak için sadece bir **gömme modeli** kullandık.

Şimdi, sorumuzla bir cevap almak için üretici bir LLM kullanacağız: ona aldığımız belgeler ve sorumuzla besleyeceğiz.

Bunu yapmanın en temel yolu tüm girdilerimizi birbirine bağlamak, sorumuzla eklemek ve sonucu görmek olacaktır.

Bir deneyelim.

👉 İlk olarak önceki meydan okumalarda olduğu gibi bir LLM başlatın.

In [51]:
from langchain.chat_models import init_chat_model

llm = init_chat_model("gemini-2.5-flash-lite", model_provider="google_genai")


Sonra temel bir istem oluşturun:

In [52]:
prompt = '\n\n'.join([doc.page_content for doc in retrieved_docs])
prompt += "\n\n" + query

👉 Şimdi istemi kullanın:

In [53]:
response = llm.invoke(prompt)
Markdown(response.content)

The provided text doesn't directly discuss agricultural policy. However, it mentions a few points that are indirectly related to agriculture and food security:

*   **European solidarity: key to securing medicine access in a shifting geopolitical era (debate)** (Item 16): While the focus is on medicines, the underlying theme of securing essential resources in a changing geopolitical landscape can be applied to food as well. A strong and stable food supply chain is crucial for any nation, and geopolitical shifts can impact this.
*   **The multiannual plan for the Baltic Sea and ways forward (debate)** (Item 15): This discussion touches on fisheries, which is a part of the broader food production sector. The text mentions improving the income of fishers and reducing fluctuations, which are economic considerations relevant to food producers.
*   **The impact of the implementation of the Maritime Spatial Planning Directive 2014/89/EU on fisheries in selected fishing areas and sea basins (debate)** (Item 14): Again, this is about fisheries, a component of food production.

In summary, while there isn't a dedicated discussion on agricultural policy, the text provides glimpses into related areas like food production (fisheries) and the broader concept of securing essential resources in a complex geopolitical environment.

Bu fena değil, ama modele daha fazla rehberlik vererek daha kapsamlı bir istem yazarak daha iyisini yapabiliriz.

Bunu yapan ilk kişiler biz değilmişiz ve LangChain'in bizim için önceden hazırlanmış istem kütüphanesi var.

👉 Aşağıdaki hücreyi çalıştırın ve nasıl çalıştığını anlamaya çalışın. (LangSmithMissingAPIKeyWarning hakkında bir uyarı alacaksınız, bunu görmezden gelebilirsiniz.)

In [54]:
from langchain_classic import hub

prompt_template = hub.pull("rlm/rag-prompt")

example_messages = prompt_template.invoke(
    {"context": "(context goes here)", "question": "(question goes here)"}
).to_messages()

print("\n")
print(example_messages[0].content)



You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.
Question: (question goes here) 
Context: (context goes here) 
Answer:


/home/ubtuna/.pyenv/versions/langchain-env/lib/python3.12/site-packages/langsmith/client.py:7943: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit list of allowed classes (or 'messages' for untrusted input that contains only chat messages) to suppress this warning.
  prompt = loads(json.dumps(prompt_object.manifest))


LangChain'in bizim için nasıl daha kesin bir istem oluşturduğunu görüyor musunuz? Bunu RAG'ımız için kullanalım!

👉 İlk olarak, tüm alınan belgeleri iki yeni satırla ayrılmış tek bir uzun dizgiye birleştirin.

In [55]:
docs_content = "\n\n".join(doc.page_content for doc in retrieved_docs)
Markdown(docs_content)

10 - Corrigenda (Rule 251) .......................................................................................................29 
11 - Signature of acts adopted in accordance with the ordinary legislative procedure (Rule 81)
 ...................................................................................................................................................30 
12 - Order of business ...............................................................................................................30 
13 - Negative trade-related effects of global overcapacity on the Union steel market (debate)
 ...................................................................................................................................................36 
14 - The impact of the implementation of the Maritime Spatial Planning Directive 
2014/89/EU on fisheries in selected fishing areas and sea basins (debate) ..............................56 
15 - The multiannual plan for the Baltic Sea and ways forward (debate) ................................69 
16 - European solidarity: key to securing medicine access in a shifting geopolitical era 
(debate) .....................................................................................................................................79 
17 - Research Fund for Coal and Steel: research programme and multiannual technical and 
financial guidelines (short presentation) ...................................................................................92 
18 - One-minute speeches on matters of political importance ..................................................95 
19 - Agenda of the next sitting ................................................................................................105 
20 - Approval of the minutes of the sitting .............................................................................105

is a critical part of our economy and it is essential that we preserve a strong and sustainable EU 
steel industry. 
 
However, in recent times, this industry has suffered from unsustainable levels of global 
overcapacity. To illustrate this problem, by next year, this overcapacity is expected to reach 
721 million tonnes, more than five times the EU's annual steel consumption. This has already 
resulted in the EU steel industry shedding 65 million tonnes in production and as many as 
100 000 jobs over the past two decades. 
 
Unfortunately, our efforts to address the problem of overcapacity with like‑minded countries in 
the context of the Global Forum on Steel Excess Capacity have not delivered satisfactory results. 
Therefore, we have had to resort to an autonomous measure to provide effective protection to our 
steel industry after the expiry of the current safeguard measure on 30 June 2026. 
 
I welcome the cooperative approach which resulted in political agreement on this steel measure. A 
timely adoption and entry into force on 1 July 2026 is now essential to ensure that there is no gap 
in protection for our industry. 
 
Such a gap would inevitably result in a massive surge of imports of steel into the European Union, 
leading to stagnating prices, huge stockpiling and ultimately to mass closures on steel producing
18-05-2026  37 
sites. This would have serious detrimental effects on the EU's steel industry and more generally on 
our overall industrial policy. Therefore, I trust the European Parliament will strongly support the 
text. 
 
Let me stress that we are adopting this measure in a way which is consistent with our international 
trade obligations. Given that some 80 % of our steel imports come from partners with whom we 
have concluded free trade agreements, they must be covered by the scope of the measure to ensure 
its effectiveness. 
 
This is not a decision that was taken lightly. However, from the beginning of the process, the EU

marine protected areas. 
 
Additionally, it touches on issues like nutrient inflows, eutrophication, pollutants, unexploded 
ammunition, offshore wind farms or the behaviour of Russia, to name just a few.
18-05-2026  71 
I fully share your view that these issues must be addressed. At the same time, many of the 
challenges we face in the Baltic Sea go beyond fisheries, and the common fisheries policy is not the 
only instrument for addressing those challenges. A more comprehensive and holistic approach, 
with coordinated cross-border action across all relevant sectors and policy fields is needed. This is 
precisely why the European Ocean Pact is so critical to bring science, policy and practice together 
to build a comprehensive strategy that delivers real results. 
 
Other key instruments to address the challenges are part of the wider context of the CFP, such as 
the Control Regulation, improving scientific advice, the internal distribution of national quotas, 
quota flexibility or the protection of certain species. These important topics will be addressed in 
the follow-up to the evaluation of the Common Fisheries Policy Regulation, including the 
preparation of the Vision 2040 for fisheries and aquaculture planned for this autumn. 
 
More specifically on the multiannual plan for the Baltic Sea, the report includes a call on the 
Commission to evaluate and possibly revise it. We will carefully look into all options. 
 
I would like to recall the joint statement by the Commission and the Baltic Member States agreed 
at the AGRIFISH Council last October, which confirmed our common ambition and 
determination to start rebuilding trajectories for the fisheries in the Baltic Sea, which would 
improve the income of fishers and reduce the fluctuations between years. We now need to 
confirm this path at the October Council this year. 
 
As regards science, my services have already been working with scientists and the Member States

related to many European countries. So we do not want to speak against. 
 
We would like to ask, perhaps, if Mr Kobosko would allow this debate to have a wider context and 
not only speak about Poland, but to speak about the situation in Europe as a whole. So we 
propose: 'Europe's response to reduced US military deployment in Europe, particularly at its 
eastern flank, in the light of the latest US decision'.
18-05-2026  33 
If Mr Kobosko would allow that, we could have a good debate that takes on all the elements that 
are relevant and all the decisions taken over the past months. 
 
1-0038-0000 
President. – Thank you, Mr Lenaers. Mr Kobosko, do you accept the proposal by Mr Lenaers to 
amend it as such? And then we vote on it. 
 
1-0039-0000 
Michał Kobosko (Renew). – Yes, it's accepted by me. 
 
1-0040-0000 
President. – Good. So we will put that request to a vote by roll call. 
 
It's adopted. Now the sitting will be extended to 23:00. 
 
So, for tomorrow, The Left Group has requested that a statement by the Vice-President / High-
Representative on 'Israel's latest airstrikes in Lebanon, the urgent need to uphold the ceasefire and 
the EU's responsibility to prevent further escalation' be added as the fourth item in the afternoon. 
 
I give the floor to Ms Rima Hassan to move the request on behalf of The Left Group. 
 
1-0041-0000 
Rima Hassan, au nom du groupe The Left. – Madame la Présidente, chers collègues, entre le moment 
de l'adoption de l'agenda et aujourd'hui, il y a eu une prolongation du cessez-le-feu entre Israël et 
le Liban de 45 jours, donc vendredi. Et il se trouve que, hier, ce cessez-le-feu a été de nouveau 
violé, faisant sept morts du côté libanais. C'est la raison pour laquelle nous vous demandons de 
reconsidérer la situation au Liban en acceptant qu'il y ait débat. 
 
1-0042-0000 
President. – Now, I give the floor to anyone who would like to speak against. No one. We put the 
request to a vote by roll call.

👉 Sonra, sorgunuz ve alınan belgelerden başlayarak bir `prompt` oluşturun. Yukarıdaki örneğe bakmayı unutmayın.

In [56]:
prompt = prompt_template.invoke(
    {"context": docs_content, "question": query}
)

👉 Son olarak az önce oluşturduğumuz `the_prompt` ile LLM modelini kullanın:

In [57]:
answer = llm.invoke(prompt)

In [58]:
Markdown(answer.content)

The provided text does not contain information about agricultural policy. The discussions revolve around the EU steel industry, maritime spatial planning and fisheries, and geopolitical matters.

🎉 İlk RAG'ımızı tamamladık: LLM kendisine sağladığımız belgelerde ***temelli*** metin üretti.

## 💾 Gömmelerimizi kalıcı hale getir

Şimdiye kadar bellekte vektör deposuyla çalıştık. Bu yüzden not defterinizi kapattığınızda, tüm gömmeleri de kaybedeceksiniz.

⚠️ Bu gömmelerin sağlayıcınızın platformunda, bu durumda Google'ın makinelerinde çalışan modeller tarafından üretildiğini unutmayın. Ve bedava çalışmazlar. 💰

Bunun gibi bir, nispeten küçük belge için maliyet düşüktür, ama hızla artar. Şimdiye kadar sadece bir günün transkriptleriyle çalıştık. Oturum başına 3 tane daha, yılda 12 oturum, birden fazla yıl var...

Bunu çözmek için sadece vektör depomuzla kalıcı bir taneyi değiştireceğiz. Bu LangChain'in avantajıdır: bileşenleri değiştirmek çok kolay.

Bellekteki vektör depomuz deneme için harikaydı, şimdi başka bir taneyle değiştireceğiz. Çok popüler bir vektör deposu olan [Chroma](https://www.trychroma.com/)'yı kullanacağız. Bunu yerel olarak çalıştırabilir ve LangChain aracılığıyla kullanabiliriz.

Tüm akışımızı yeniden oluşturacağız. Her şeyi birkaç kod hücresinde tekrar bir araya getirmeye çalışmak iyi bir alıştırmadır. Aynı zamanda her şeyi yeniden kullanılabilir koda dönüştüreceğiz.

Sonunda iki fonksiyon istiyoruz:

1. `embed_and_store()`: Başka bir oturumun transkriptini vektör veritabanımıza ekle, böylece alacağımız daha fazla veri olsun.
2. `answer()`: Vektör depomuzla farklı sorularla sorgula.

#### 1. Bir Chroma vektör deposu başlat

👉 **Veri kalıcılığıyla** (yani verileri diskteki bir dizinde saklayarak) Chroma vektör deposunun nasıl oluşturulacağını görmek için [LangChain'in belgelerine](https://python.langchain.com/docs/integrations/vectorstores/chroma/) bakın.

In [68]:
from langchain_google_vertexai import VertexAIEmbeddings

embeddings = VertexAIEmbeddings(
    model_name="text-embedding-005",
    project="project-ef14cbd0-0f6d-42bb-b6d",
    location="us-central1"
)

from langchain_chroma import Chroma

vector_store = Chroma(
    collection_name="ep_plenary",
    embedding_function=embeddings,
    persist_directory="./chroma_ep_follower",
)

/tmp/ipykernel_43749/175048281.py:3: DeprecationWarning: Use [`GoogleGenerativeAIEmbeddings`][langchain_google_genai.GoogleGenerativeAIEmbeddings] instead.
  embeddings = VertexAIEmbeddings(


#### 2. `embed_and_store()` oluştur

👉 Bu fonksiyon için kodu tamamlayın:

In [78]:
import time

def embed_and_store(file_path, vector_store):
    """Load a PDF file, split it into chunks, and store the chunks in a vector store."""
    loader = PyPDFLoader(file_path, mode='single')
    pdf = loader.load()

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=100,
        add_start_index=True,
    )
    all_splits = text_splitter.split_documents(pdf)

    batch_size = 100
    document_ids = []
    for i in range(700, len(all_splits), batch_size):
        batch = all_splits[i:i+batch_size]
        ids = vector_store.add_documents(documents=batch)
        document_ids.extend(ids)
        print(f"{i+batch_size}/{len(all_splits)} tamamlandı")
        time.sleep(30)

    print(f"Added {len(document_ids)} documents to the vector store.")
    return document_ids

👉 Fonksiyonunuzu bir dosya veya hatta iki dosyayla deneyin:

In [79]:
file_path = "data/CRE-10-2026-05-18_EN.pdf"
document_ids = embed_and_store(file_path, vector_store)

800/961 tamamlandı
900/961 tamamlandı
1000/961 tamamlandı
Added 261 documents to the vector store.


#### 3. `answer()` oluştur

👉 Bu fonksiyon için kodu tamamlayın:

In [80]:
def answer(query, vector_store, llm, prompt_template=None):
    """Answer a query using the vector store and the language model."""
    # Retrieve similar documents from the vector store
    retrieved_docs = vector_store.similarity_search(query, k=6)

    # Create the prompt
    docs_content = "\n\n".join(doc.page_content for doc in retrieved_docs)

    # If no prompt template is provided, use the default one
    if not prompt_template:
        prompt_template = hub.pull("rlm/rag-prompt")

    prompt = prompt_template.invoke(
        {"context": docs_content, "question": query}
    )

    # Get the answer from the language model
    answer = llm.invoke(prompt)

    return answer.content

👉 Fonksiyonunuzu beğendiğiniz bir sorguyla deneyin:

In [81]:
answer("What was discussed about agricultural policy?", vector_store, llm)

/home/ubtuna/.pyenv/versions/langchain-env/lib/python3.12/site-packages/langsmith/client.py:7943: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit list of allowed classes (or 'messages' for untrusted input that contains only chat messages) to suppress this warning.
  prompt = loads(json.dumps(prompt_object.manifest))


'The provided text mentions that the agricultural sector is among the first to be affected by the economic impacts of the crisis in the Middle East. It highlights that rising agricultural production costs inevitably lead to increased food prices for citizens, impacting their quality of life. The text also notes that the AGRI Committee has transmitted a corrigendum to a text adopted by Parliament.'

🏁 Tebrikler! Artık LangChain kullanarak RAG'da ustalaştınız ve vektör deponuza daha fazla belge eklemek ve onu sorgulamak için yeniden kullanılabilir fonksiyonlar yapmayı öğrendiniz.

## [İsteğe Bağlı] Meta veri ekleme

Kurduğumuz RAG, vektör deposundaki tüm belgeleri sorgular. Orada birden fazla yılın bilgisinin olduğunu düşünün. Yıllara veya tarihlere göre filtreyebilsek kullanışlı olurdu, değil mi?

Bunu nasıl yaparız? Vektör deposundaki belgelerin meta veri içerdiğini unutmayın. Eğer tarihi ekleyebilseydik, daha sonra filtrelemek için kullanabilirdik.

İpucu: Meta verilerinizi pipeline'ınızda olabildiğince erken ekleyin. Verileriniz vektör deposunda saklandıktan sonra eklemeye çalışmayın.

👉 `embed_and_store()` fonksiyonunuzu uyarlayın.

In [82]:
def embed_and_store_fancy(file_path, vector_store, session_date):
    """Load a PDF file, split it into chunks, and store the chunks in a vector store.
    Session_date is added to the metadata of each chunk."""
    loader = PyPDFLoader(file_path, mode='single')
    pdf = loader.load()

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=100,
        add_start_index=True,
    )
    all_splits = text_splitter.split_documents(pdf)

    for split in all_splits:
        split.metadata['session_date'] = session_date

    batch_size = 100
    document_ids = []
    for i in range(0, len(all_splits), batch_size):
        batch = all_splits[i:i+batch_size]
        ids = vector_store.add_documents(documents=batch)
        document_ids.extend(ids)
        time.sleep(2)

    return document_ids

👉 Fonksiyonunuzu deneyin ve vektör deponuzun ek meta veri içerdiğini kontrol edin.

In [83]:
document_ids = embed_and_store_fancy("data/CRE-10-2026-05-18_EN.pdf", vector_store, "2026-05-18")
vector_store.get_by_ids(document_ids[:3])

ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'Quota exceeded for aiplatform.googleapis.com/online_prediction_requests_per_base_model with base model: textembedding-gecko. Please submit a quota increase request. https://cloud.google.com/vertex-ai/docs/generative-ai/quotas-genai.', 'status': 'RESOURCE_EXHAUSTED'}}

Şimdi alıcıyı kullanıcının sorduğu tarihe göre sınırlamamız gerekiyor.

👉 `answer()` fonksiyonunuzu bir tarih alabilecek ve yeni meta verilere dayalı olarak belgeleri filtreleyebilecek şekilde uyarlayın.

In [84]:
def answer(query, vector_store, llm, session_date=None, prompt_template=None):
    if session_date:
        retrieved_docs = vector_store.similarity_search(
            query, k=6, filter={"session_date": session_date}
        )
    else:
        retrieved_docs = vector_store.similarity_search(query, k=6)

    docs_content = "\n\n".join(doc.page_content for doc in retrieved_docs)

    if not prompt_template:
        prompt_template = hub.pull("rlm/rag-prompt")

    prompt = prompt_template.invoke(
        {"context": docs_content, "question": query}
    )

    result = llm.invoke(prompt)
    return result.content

In [ ]:
# YOUR CODE HERE

Harika! Güçlü bir RAG sistemi oluşturmak için benzerlik aramasını meta veri aramasıyla birleştirdiniz!